# 4단계: Q1 넷째 데이터셋 — MVTec AD 2

Q1 과 같은 설정 10개 x 시드 3개를 MVTec AD 2 의 8부류에 돌려, 세 데이터셋에서 낸 순위와 견준다.

| 갈래 | 장수 | 쓰나 |
|---|---|---|
| train/good | 2,528 (부류마다 137~432), 모두 regular | 학습 |
| test_public | 1,084 (정상 379, 결함 705, 마스크 705) | 채점 |
| validation/good | 302 | 안 씀(두 방법 모두 쓰지 않음) |
| test_private, test_private_mixed | 부류마다 142~321 | 안 씀(정답 비공개) |

**정한 것(2026-11-13).**
- 전처리는 Q1 과 같은 `Resize((256, 256))`, 자르지 않음. 원본이 1400~4224 픽셀이라 256 에서 결함 덩어리 2,058개 중 483개가 사라지고, 마스크 12장(can 6, fabric 6)은 비게 된다(mvtec2_check2 노트북). 이미지 라벨은 남으므로 이미지 AUROC 를 주 지표로 본다.
- test_public 의 조명 조건(regular, overexposed, underexposed, shift_1~3, vial 만 shift_4)을 모두 함께 채점한다. 조건별 이미지 AUROC 는 5번 둘째 셀에서 점수 파일로 따로 낸다. 학습은 regular 만 있어서, 순위가 바뀌면 그 안에 조명 변화에 대한 강건성이 섞인다.

**시간(어림, 잰 값 아님).**
- 2번 캐시: 원본을 256 으로 한 번 줄여 Drive `mvtec2_256/` 에 둔다. 부류마다 10장으로 잰 첫 읽기 속도(평균 4.4초, fabric 19.8초)로 4,317장이면 약 6시간이고, 여기서는 8장씩 동시에 읽으니 그보다 짧을 수 있다. 한 번 만들면 다시 만들지 않는다.
- 4번 실행: 240번, Q1 의 MVTec AD 몫(450번, 5.9시간)을 장수 비로 줄여 약 5.1시간.

**끊겨도 된다.**
- 2번 캐시는 한 장마다 Drive 에 저장한다. 다시 돌리면 있는 파일은 건너뛰고, 다 끝난 부류는 통째로 건너뛴다.
- 4번 실행은 부류 하나가 끝날 때마다 Drive 의 `q1_mvtec_ad_2.csv` 에 한 줄을 적는다. 다시 돌리면 적힌 줄은 건너뛴다.
- 이어 돌리는 순서: 런타임 > 세션 다시 시작 -> 1번 셀 -> 2번 두 셀(캐시가 끝났으면 바로 지나간다) -> 3번 첫 셀 -> 4번 셀.

**실행 전 준비.** 런타임 GPU(L4), 보안 비밀 `GITHUB_TOKEN` 접근 허용. 02 노트북과 같다.


## 1. 코드와 패키지

In [ ]:
import os, subprocess, sys

REPO_DIR = "/content/beyond-mvtec"
REPO_URL = "https://github.com/chlee1027/beyond-mvtec.git"
UPSTREAM_DIR = "/content/patchcore-inspection"
UPSTREAM_COMMIT = "fcaa92f124fb1ad74a7acf56726decd4b27cbcad"

from google.colab import userdata
token = userdata.get("GITHUB_TOKEN").strip()
auth_url = REPO_URL.replace("https://", f"https://x-access-token:{token}@")
if not os.path.isdir(REPO_DIR):
    r = subprocess.run(["git", "clone", "-q", auth_url, REPO_DIR], capture_output=True, text=True)
else:
    # 이미 받아 둔 레포는 GitHub 의 최신 main 으로 맞춘다(토큰 든 주소는 저장하지 않는다)
    r = subprocess.run(["git", "-C", REPO_DIR, "pull", "-q", "--ff-only", auth_url, "main"],
                       capture_output=True, text=True)
# 실패해도 오류 메시지에 토큰이 찍히지 않게 가린다
assert r.returncode == 0, "레포 받기 실패: " + r.stderr.replace(token, "***")
subprocess.run(["git", "-C", REPO_DIR, "remote", "set-url", "origin", REPO_URL], check=True)
if not os.path.isdir(UPSTREAM_DIR):
    subprocess.run(["git", "clone", "-q", "https://github.com/amazon-science/patchcore-inspection.git", UPSTREAM_DIR], check=True)
subprocess.run(["git", "-C", UPSTREAM_DIR, "checkout", "-q", UPSTREAM_COMMIT], check=True)

!pip uninstall -y -q sympy
!pip install -q sympy==1.13.3 faiss-gpu-cu12 timm click pretrainedmodels scikit-image scikit-learn scipy tqdm pandas pyyaml

sys.path[:0] = [os.path.join(REPO_DIR, "src"), os.path.join(UPSTREAM_DIR, "src")]
COMMIT = subprocess.run(["git", "-C", REPO_DIR, "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip()

import faiss, torch, torchvision
assert torch.cuda.is_available(), "GPU 런타임이 아니다"
print(f"레포 {COMMIT} | Python {sys.version.split()[0]} | torch {torch.__version__} | torchvision {torchvision.__version__} | faiss {faiss.__version__}")
print(f"GPU {torch.cuda.get_device_name(0)} | VRAM {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
assert hasattr(faiss, "StandardGpuResources"), "faiss 의 GPU 판이 아니다"

from beyond_mvtec import data, experiment as ex, patchcore_adapter
config = ex.load_config(f"{REPO_DIR}/configs/experiments.yaml")
SETTINGS = ex.comparison_settings(config)
# 레포가 MVTec AD 2 코드보다 옛 판이면 여기서 멈춘다
assert len(SETTINGS) == 10 and hasattr(data, "list_mvtec2") and hasattr(data, "cache_mvtec2") \
    and "mvtec_ad_2" in config["datasets"] and hasattr(patchcore_adapter, "_releasing_sampler"), \
    "옛 코드가 읽혔다. 런타임 > 세션 다시 시작 후 1번 셀부터 다시 돌린다"
print("설정", len(SETTINGS), "개:", [s["setting"] for s in SETTINGS])

## 2. 256 캐시와 데이터

첫 셀이 Drive `mvtec2/` 의 원본을 `mvtec2_256/` 으로 줄여 저장하고(부류마다 진행 표시), 다 된 캐시를 런타임 디스크로 옮긴다. 둘째 셀은 장수와 조건별 장수를 점검 노트북에서 센 값과 맞춰 보고, 부류마다 원본 세 장을 직접 읽은 텐서와 캐시를 읽은 텐서가 같은지 본다.

In [ ]:
import shutil, time
from google.colab import drive
drive.mount("/content/drive")

MY = "/content/drive/MyDrive"
os.environ["TORCH_HOME"] = f"{MY}/torch_cache"
RESULT_DIR = f"{MY}/beyond_mvtec_results"
os.makedirs(RESULT_DIR, exist_ok=True)

PREPROCESS = config["preprocess"]["compare"]
SIZE = config["datasets"]["mvtec_ad_2"]["cache"]
# 캐시 크기와 실행 전처리가 같아야 원본을 읽은 것과 값이 같다
assert list(PREPROCESS["resize"]) == list(SIZE) and not PREPROCESS.get("center_crop"), (PREPROCESS, SIZE)

SRC_DRIVE = f"{MY}/mvtec2"
CACHE_DRIVE = f"{MY}/mvtec2_256"
LOCAL = "/content/data/mvtec_ad_2"
CATEGORIES = data.MVTEC2_CATEGORIES
assert all(os.path.isdir(f"{SRC_DRIVE}/{c}") for c in CATEGORIES), os.listdir(SRC_DRIVE)

for category in CATEGORIES:
    start = time.time()
    n = data.cache_mvtec2(SRC_DRIVE, CACHE_DRIVE, category, SIZE, workers=8)
    print(f"{category}: 새로 쓴 장수 {n}, {time.time() - start:.0f}초")

if not os.path.isfile(f"{LOCAL}/.copied"):
    shutil.copytree(CACHE_DRIVE, LOCAL, dirs_exist_ok=True)
    open(f"{LOCAL}/.copied", "w").close()
print("Drive:", CACHE_DRIVE, "-> 로컬:", LOCAL)


In [ ]:
import numpy as np
import pandas as pd
import torch

rows, conds = [], []
for category in CATEGORIES:
    split = data.list_mvtec2(LOCAL, category)
    rows.append({"category": category, "train": len(split.train), "test": len(split.test),
                 "test_anomaly": sum(s.label for s in split.test)})
    conds += [{"category": category, "label": s.label, "condition": s.anomaly} for s in split.test]
counts = pd.DataFrame(rows).set_index("category")
display(counts)

# mvtec2_check · mvtec2_check2 노트북에서 센 값
expected = {"can": (412, 162, 90), "fabric": (387, 156, 90), "fruit_jelly": (263, 80, 60),
            "rice": (313, 132, 90), "sheet_metal": (137, 114, 90), "vial": (291, 140, 105),
            "wallplugs": (293, 150, 90), "walnuts": (432, 150, 90)}
got = {c: tuple(int(v) for v in r) for c, r in counts[["train", "test", "test_anomaly"]].iterrows()}
assert got == expected, got
display(pd.DataFrame(conds).pivot_table(index="category", columns="condition", values="label",
                                        aggfunc="size", fill_value=0))

# 캐시를 읽은 텐서가 원본을 읽은 텐서와 같은지, 부류마다 학습 한 장 · 결함 한 장 · 그 마스크로 본다
image_tf, mask_tf, output_size = data.build_transforms(PREPROCESS)
for category in CATEGORIES:
    orig, cached = data.list_mvtec2(SRC_DRIVE, category), data.list_mvtec2(LOCAL, category)
    a = data.TrainDataset(orig.train[:1], image_tf)[0]["image"]
    b = data.TrainDataset(cached.train[:1], image_tf)[0]["image"]
    i = next(k for k, s in enumerate(orig.test) if s.label == 1)
    ta = data.TestDataset(orig.test[i:i + 1], image_tf, mask_tf, output_size)[0]
    tb = data.TestDataset(cached.test[i:i + 1], image_tf, mask_tf, output_size)[0]
    same = torch.equal(a, b) and torch.equal(ta["image"], tb["image"]) and torch.equal(ta["mask"], tb["mask"])
    print(category, "같다" if same else "다르다")
    assert same, category


## 3. 짧은 실행: 가장 무거운 칸 먼저

학습이 가장 많은 `walnuts`(432장), 시드 0에서 두 설정을 먼저 돈다. 결과는 4번과 같은 CSV에 적히므로 4번에서 다시 돌지 않는다.

- `patchcore_wrn50_L123_c1`: Q1 에서 MVTec AD `hazelnut`(391장)의 `fit_sec` 가 204.6~208.3초였다. 코어셋 시간은 장수의 제곱에 비례하므로 432장이면 약 252초로 어림한다.
- `padim_wrn50_L123_d550`: 같은 256 해상도라 위치 수가 같다. Q1 MVTec AD 최대 GPU 메모리는 17.83 GB였다.

**넘어가는 기준.** 둘 다 오류 없이 끝나고, PatchCore의 `fit_sec`가 200~460초 안이면(어림의 0.8~1.8배, 02 노트북과 같은 폭) 4번으로 바로 간다. 메모리 오류가 나거나 `fit_sec`가 이 범위를 벗어나면 4번을 돌리지 말고 이 셀 출력을 보낸다.

In [ ]:
import resource

OPTIONS = dict(device="cuda", batch_size=8, num_workers=4, faiss_on_gpu=True, commit=COMMIT,
               save_dir=f"{RESULT_DIR}/q1_scores")
CSV = f"{RESULT_DIR}/q1_mvtec_ad_2.csv"


In [ ]:
heavy = [s for s in SETTINGS if s["setting"] in ("patchcore_wrn50_L123_c1", "padim_wrn50_L123_d550")]
ex.run_grid(heavy, "mvtec_ad_2", LOCAL, ["walnuts"], [0], "compare", PREPROCESS, CSV, **OPTIONS)

done = pd.read_csv(CSV)
display(done[(done.category == "walnuts") & (done.seed == 0) & done.setting.isin([s["setting"] for s in heavy])]
        [["setting", "image_auroc", "pixel_auroc", "pro", "bank_size", "fit_sec", "predict_sec", "metric_sec", "peak_gpu_gb"]])
print(f"호스트 메모리 최대 {resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6:.1f} GB")


## 4. 실행

시드 0 -> 1 -> 2 순서로, 시드마다 설정 10개 x 8부류를 돈다. 점수와 라벨은 `q1_scores/`에 남기고(5번 조건별 표에 쓴다) 이상 지도는 남기지 않는다.

In [ ]:
for seed in config["seeds"]:
    print(f"===== 시드 {seed} | mvtec_ad_2 =====")
    ex.run_grid(SETTINGS, "mvtec_ad_2", LOCAL, CATEGORIES, [seed], "compare", PREPROCESS, CSV, **OPTIONS)


## 5. 순위와 순위 상관

레포 `results/` 의 Q1 세 CSV 와 이번 CSV 를 함께 읽어, 데이터셋 넷의 순위와 여섯 쌍의 Kendall tau-b 를 낸다. 시드 0만 끝난 상태에서도 돌릴 수 있다.

In [ ]:
paths = [f"{REPO_DIR}/results/q1_{d}.csv" for d in ["mvtec_ad", "real_iad", "visa"]] + [CSV]
summary = ex.summarize(paths)
new = summary[summary.dataset == "mvtec_ad_2"]
display(new[["setting", "image_auroc_mean", "image_auroc_std", "pixel_auroc_mean", "pro_mean",
             "n_seeds", "n_categories"]].round(2))
print("칸이 다 찼나:", ((new.n_seeds == 3) & (new.n_categories == len(CATEGORIES))).sum() == len(SETTINGS))

values, ranks = ex.rank_by_dataset(paths)
print("이미지 AUROC (%, 시드 평균)")
display(values.round(2))
print("순위 (1 = 가장 높음)")
display(ranks)
print("Kendall tau-b, 시드 평균 값으로")
display(ex.kendall_pairs(values).round(3))
print("Kendall tau-b, 시드 하나씩")
display(ex.kendall_per_seed(paths).round(3))

df = pd.read_csv(CSV)
display(df.groupby("setting")[["fit_sec", "predict_sec", "metric_sec", "peak_gpu_gb"]].mean().round(1))
print("더한 시간(시간):", round((df.fit_sec + df.predict_sec + df.metric_sec).sum() / 3600, 1))


In [ ]:
# 조건별 이미지 AUROC: 그 조건의 정상과 결함만으로 낸다. 부류 평균 -> 시드 평균, 단위 %
from sklearn.metrics import roc_auc_score

rows = []
splits = {c: data.list_mvtec2(LOCAL, c) for c in CATEGORIES}
for r in pd.read_csv(CSV).itertuples():
    stem = f"mvtec_ad_2__{r.category}__{r.setting}__compare__nall__s{r.seed}"
    saved = np.load(f"{RESULT_DIR}/q1_scores/{stem}.npz")
    test = splits[r.category].test
    labels = np.array([s.label for s in test])
    assert np.array_equal(labels, saved["labels"]), stem  # 저장 순서와 목록 순서가 같다
    cond = np.array([s.anomaly for s in test])
    for c in sorted(set(cond)):
        pick = cond == c
        rows.append({"setting": r.setting, "category": r.category, "seed": r.seed, "condition": c,
                     "image_auroc": roc_auc_score(labels[pick], saved["scores"][pick])})
by_cond = pd.DataFrame(rows)
table = (by_cond.groupby(["setting", "condition", "seed"]).image_auroc.mean()
         .groupby(["setting", "condition"]).mean().unstack() * 100)
display(table.round(2))
print("조건마다 든 부류 수:", by_cond.groupby("condition").category.nunique().to_dict())
print("regular 만으로 낸 순위와 전체 순위의 Kendall tau-b")
display(ex.kendall_pairs(pd.DataFrame({"all": values["mvtec_ad_2"], "regular": table["regular"],
                                       "mvtec_ad": values["mvtec_ad"]})).round(3))


## 6. 결과를 레포로

Drive `beyond_mvtec_results/`의 `q1_mvtec_ad_2.csv`를 레포 `results/`에 넣는다. `q1_scores/`와 `mvtec2_256/`은 올리지 않는다.